# Gabay custom models

Train two models for Gabay on Google Colab (free GPU: Runtime > Change runtime type > T4 GPU):

1. **Custom street detector**: things EfficientDet doesn't know, like `stairs`, `curb`, `hole`, `open_canal`, `pole`, `tricycle`, `jeepney`. Output: `custom_detector.tflite` + `custom_detector_labels.txt`.
2. **Peso bill classifier**: `20`, `50`, `100`, `200`, `500`, `1000` (plus `none`). Output: `peso.tflite` + `peso_labels.txt`.

Both use **MediaPipe Model Maker**, so the models plug straight into Gabay (same format as the built-in detector).

**Labels Gabay understands:** hazards (`stairs`, `stairs_down`, `stairs_up`, `curb`, `hole`, `open_canal`, `manhole`, `pole`, `wire`, `step`) are announced as dangers; vehicles (`tricycle`, `jeepney`, `e-trike`, `pedicab`) get the vehicle warning. Use these exact names.

In [ ]:
!pip install -q mediapipe-model-maker

## 1. Custom street detector

**Collect data:** take 150-300 photos per label around Malolos at the height of a chest lanyard (day, night, rain). Label boxes in **Roboflow** (free) and export as **COCO JSON**. Upload the zip here or link Google Drive.

Folder layout expected:
```
street_data/train/images/*.jpg + labels.json
street_data/validation/images/*.jpg + labels.json
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/gabay/street_data'   # change to your folder

In [ ]:
from mediapipe_model_maker import object_detector

train = object_detector.Dataset.from_coco_folder(f'{DATA}/train', cache_dir='/tmp/od_train')
val = object_detector.Dataset.from_coco_folder(f'{DATA}/validation', cache_dir='/tmp/od_val')
print('train', train.size, 'val', val.size, 'labels', train.label_names)

options = object_detector.ObjectDetectorOptions(
    supported_model=object_detector.SupportedModels.MOBILENET_MULTI_AVG,
    hparams=object_detector.HParams(epochs=60, batch_size=8, export_dir='exported_street'),
)
model = object_detector.ObjectDetector.create(train_data=train, validation_data=val, options=options)
loss, coco_metrics = model.evaluate(val, batch_size=4)
print('Validation COCO metrics (report these in Chapter 4):', coco_metrics)

In [ ]:
# Quantize to int8 for speed on phones, then export
from mediapipe_model_maker import quantization
model.restore_float_ckpt()
model.quantization_aware_training(train, val, qat_hparams=object_detector.QATHParams(learning_rate=0.3, batch_size=4, epochs=10, decay_steps=6, decay_rate=0.96))
model.export_model('custom_detector.tflite')
with open('exported_street/custom_detector_labels.txt', 'w') as f:
    f.write('\n'.join(train.label_names))
print(open('exported_street/custom_detector_labels.txt').read())

## 2. Peso bill classifier

Take about 100 photos per bill (front and back, folded, different lights and backgrounds) plus ~100 photos of random non-bill things for `none`. Folder per label:
```
peso_data/20/*.jpg  peso_data/50/*.jpg ... peso_data/1000/*.jpg  peso_data/none/*.jpg
```

In [ ]:
from mediapipe_model_maker import image_classifier
PESO = '/content/drive/MyDrive/gabay/peso_data'   # change to your folder
data = image_classifier.Dataset.from_folder(PESO)
train, rest = data.split(0.8)
val, test = rest.split(0.5)
options = image_classifier.ImageClassifierOptions(
    supported_model=image_classifier.SupportedModels.EFFICIENTNET_LITE0,
    hparams=image_classifier.HParams(epochs=20, export_dir='exported_peso'),
)
model = image_classifier.ImageClassifier.create(train_data=train, validation_data=val, options=options)
loss, acc = model.evaluate(test)
print(f'Test accuracy (report this): {acc:.3f}')
model.export_model('peso.tflite')
with open('exported_peso/peso_labels.txt', 'w') as f:
    f.write('\n'.join(data.label_names))
print(open('exported_peso/peso_labels.txt').read())

## 3. Put the models in Gabay

Download from the Colab file panel and add them to the GitHub repo:

```
assets/models/custom_detector.tflite
assets/labels/custom_detector_labels.txt
assets/models/peso.tflite
assets/labels/peso_labels.txt
```

Commit and push. The GitHub build includes them automatically, and Settings > AI engines will show them as loaded.